# Pump It Up model comparison

Define the first feature policy and prove that its learned preprocessing fits within a development fold. This first increment deliberately stops before fitting a feature-based model.

The notebook keeps the decisions visible while delegating mechanical feature engineering and preprocessing to `src/feature_engineering.py` and `src/model_preprocessing.py`. It reconstructs the frozen design from `02-baseline.ipynb`; the local test and competition rows remain outside this smoke test.

In [1]:
from pathlib import Path
import sys

import pandas as pd

NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name != 'notebooks' or NOTEBOOK_DIR.parent.name != 'stage-1-pump-it-up':
    raise RuntimeError(
        'Run this notebook from the stage-1-pump-it-up/notebooks directory.'
    )

STAGE_DIR = NOTEBOOK_DIR.parent
DATA_DIR = STAGE_DIR / 'data'
SRC_DIR = STAGE_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from data_partitioning import make_cross_validation, partition_modelling_data
from feature_engineering import summarise_initial_feature_policy
from model_preprocessing import smoke_test_first_fold
from modelling_data import prepare_modelling_data

## Reconstruct the frozen development design

Load the immutable source files, apply the settled structural preparation and recreate the fixed local test membership and five development folds. No cleaned copy is written.

In [2]:
raw_original = pd.read_csv(DATA_DIR / 'TrainingSetValues.csv')
labels_original = pd.read_csv(DATA_DIR / 'TrainingSetLabels.csv')
raw_competition = pd.read_csv(DATA_DIR / 'TestSetValues.csv')

modelling_data = prepare_modelling_data(
    raw_original,
    labels_original,
    raw_competition,
)
partitioned_data = partition_modelling_data(modelling_data)
cross_validation = make_cross_validation(partitioned_data)

## Fix the initial feature policy

`date_recorded` becomes `days_since_recorded`; it is not split into year, month or day fields. The fixed reference is **2 February 2015**, the date of the [earliest surviving official Pump It Up community post](https://community.drivendata.org/t/about-the-pump-it-up-data-mining-the-water-table-category/63). DrivenData no longer exposes a verified original launch date, so the code labels this honestly as a competition-era reference date. All supplied recording dates precede it.

The first pass also favours a compact, interpretable feature set. High-cardinality identifiers and alternate levels of deterministic category hierarchies are deferred for later ablations rather than allowed to dominate the first model.

In [3]:
print(summarise_initial_feature_policy().to_string())

                                                                                                                                                       treatment
feature_group                                                                                                                                                   
Recording date                                                                                    Replace date_recorded with days_since_recorded from 2015-02-02
Construction year                                                                         Replace with pump_age_at_recording plus missing and inconsistent flags
Measurement sentinels                                        Flag unavailable height, coordinates and population; leave fold-fitted median imputation downstream
Categorical predictors                                                               Use explicit missing values, fold-fitted rare grouping and one-hot encoding
Deferred high-cardinality         

## Smoke-test preprocessing on one fold

Use fold 1 as validation. Deterministic feature engineering runs inside the pipeline, then numeric medians, missing-value indicators, rare-category grouping and one-hot categories are learned from the other four folds only. Transform fold 1 without refitting.

The helper also sends a one-row synthetic unseen category through the fitted pipeline. This checks the fallback path even if every naturally occurring validation level happened to appear in the training rows.

In [4]:
preprocessing_smoke = smoke_test_first_fold(
    partitioned_data,
    cross_validation,
)
initial_preprocessor = preprocessing_smoke.preprocessor

print(preprocessing_smoke.summary.to_string())

unseen_validation_categories = preprocessing_smoke.categorical_coverage.query(
    'unseen_validation_levels > 0'
)
if unseen_validation_categories.empty:
    print('All selected categorical levels in fold 1 also occur in its training rows.')
else:
    print(unseen_validation_categories.to_string())

                                                 value
measurement                                           
Validation fold                                      1
Training rows                                    38016
Validation rows                                   9504
Raw predictors received                             36
Engineered predictors selected                      29
Transformed predictors                             301
Sparse output                                     True
Synthetic unseen category handled                 True
Competition-era reference date              2015-02-02
Training days_since_recorded range    426 to 4494 days
Validation days_since_recorded range  426 to 3990 days
Non-finite transformed values                        0
All selected categorical levels in fold 1 also occur in its training rows.


### Interpretation

The fold-fitted pipeline accepts all 36 prepared predictors, selects and engineers 29 initial predictors, and produces a finite sparse matrix for both training and validation rows. `date_recorded` is absent after engineering; the only direct recording-time feature is elapsed days from the fixed competition-era reference.

The local test and competition data have still not been transformed, inspected or scored.

## Next step

Wrap this preprocessor and a constrained decision tree in one scikit-learn pipeline. Evaluate that complete pipeline across all five frozen development folds and compare its accuracy and per-class recall with the 54.31% majority reference.